# NLP HW02 shape bench

Один offline прогон на одной T4, ≤20 минут. Случайные веса, без полезного обучения и сохранения checkpoints.
13 форм; контроль C6+6/d512. 8 одинаковых effective batches128 с реальными joint lengths из verified cache v2; AMP/fused AdamW, longest-row micro probe64→1, warmup вне таймеров. Обновления нужны только для измерения forward/backward/optimizer. Перед decode исходные random weights восстановлены.

Frozen v6 decoder all (compaction, preallocated KV, cross projection, race sampler); natural EOS запрещён, завершение принудительно по общему распределению длин L3 C-main. Greedy1024 вопросов; sampling256×K4=1024 последовательности. OOM уменьшает lanes с явной записью. Encoder отдельно1919 вопросов, batch256, 3 повтора. Только длины, текст/targets не используются. Это сравнение форм; качество и полный production timer не измеряются.

Входы: thisarivson/nlp-hw02-26b-inputs, pinned CPU kernel thisarivson/nlp-hw02-compact-cache-verified/2. Длины уже встроены из этого кеша; монтирования не читаются, повторной сборки кеша нет. Internet off, без pip и API. Working deadline1100с, watchdog1150с, stop1180с; partial честно сохраняется при недостающей строке. Скачайте shape-bench/summary.json, timings.csv и run.log.

In [ ]:
import json, os, signal, subprocess, sys, threading, time
from pathlib import Path
SESSION_START=time.monotonic()
ROOT=Path('/kaggle/working/shape-bench-modules'); ROOT.mkdir(exist_ok=True)
OUTPUT=Path('/kaggle/working/shape-bench'); OUTPUT.mkdir(exist_ok=True)
stop=threading.Event(); children=set()
def kill_children():
    for pid in list(children):
        try: os.killpg(pid,signal.SIGKILL)
        except ProcessLookupError: pass
def watch():
    while not stop.wait(.2):
        if time.monotonic()-SESSION_START>=1150:
            kill_children()
            (OUTPUT/'watchdog.json').write_text(json.dumps({'triggered':True,'wall_s':time.monotonic()-SESSION_START}))
            return
threading.Thread(target=watch,daemon=True).start()

SOURCES={'final_core.py': "# Generated by build_final_inference.py from exp26_core.py; do not edit.\nimport math,time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError('Position overflow')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b''.join(chunks)\n\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n\nclass BudgetExpired(Exception):pass\n\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired('Session budget reached; completed run boundaries are preserved')\n\n\ndef sync(device):\n if device.type=='cuda':torch.cuda.synchronize(device)\n\n", 'final_fast.py': '# Generated by build_final_inference.py from exp27b_fast.py; do not edit.\nimport math\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            self.cross.append(tuple(t.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous() for t in (k,v)))\n\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n\nfrom final_decode import decode\n', 'final_decode.py': '"""V3 decode loop; frozen Context/Decoder mathematics live in final_fast."""\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, check_every=16, compute_logprob=True):\n    from final_fast import Decoder\n    d = Decoder(context, k)\n    device = context.src.device\n    token = torch.full((d.n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(d.n, dtype=torch.bool, device=device)\n    # KV, output, running LP/length buffers are allocated once, never concatenated.\n    output = torch.full((d.n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums = torch.zeros(d.n, device=device)\n    lengths = torch.zeros(d.n, dtype=torch.long, device=device)\n    for t in range(context.capacity):\n        if t % check_every == 0:\n            C.check_deadline(deadline)\n        logits = d.step(token)\n        if temperature is not None or compute_logprob:\n            logits = logits.float()\n        logits[:, :2] = -torch.inf  # PAD/BOS: contiguous slice avoids a per-step index tensor.\n        pick = logits.argmax(-1) if temperature is None else torch.multinomial(\n            F.softmax(logits / temperature, -1), 1).squeeze(-1)\n        if compute_logprob:\n            logprob = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n            sums.add_(torch.where(done, 0., logprob))\n        lengths.add_(~done)\n        token = pick.masked_fill(done, C.PAD)\n        output[:, t] = token\n        done |= token.eq(C.EOS)\n        if (t + 1) % check_every == 0 and bool(done.all().cpu()):\n            break\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (sums / lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_speed.py': '"""Opt-in decoder experiments. Frozen final_fast/final_decode remain the control."""\nfrom dataclasses import dataclass\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\nimport final_fast as Fast\n\n\nGRAPH_REFUSAL = (\n    \'CUDA graphs disabled: prefix length, KV slice addresses and active batch change every step. \'\n    \'One graph per position/batch would require hundreds of captures; full-capacity masked SDPA \'\n    \'changes reduction shapes and cannot promise bit-exact greedy. No untested graph fallback.\')\n\n\n@dataclass(frozen=True)\nclass Options:\n    preallocated_kv: bool = True  # Already present in the frozen baseline; no new speedup.\n    share_cross: bool = False\n    async_eos: bool = False\n    eos_every: int = 16\n    compact_done: bool = False\n    compact_fraction: float = .25\n    race_sampler: bool = False\n    lean_greedy: bool = False\n    cuda_graphs: bool = False\n\n    def __post_init__(self):\n        if not self.preallocated_kv:\n            raise ValueError(\'Baseline already preallocates KV; a cat regression is not an optimization\')\n        if self.cuda_graphs:\n            raise NotImplementedError(GRAPH_REFUSAL)\n        if self.eos_every < 1 or not 0 < self.compact_fraction <= 1:\n            raise ValueError(\'Invalid EOS/compaction interval\')\n\n\nclass Decoder(Fast.Decoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        if not share_cross or k != 1:\n            super().__init__(context, k)\n            return\n        # Immutable cross KV and source mask can be shared for greedy, no replication.\n        self.c, self.k, self.t, self.n = context, k, 0, len(context.src)\n        self.allowed, self.cross = context.allowed, context.cross\n        self.cross_bytes = sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [(context.positions.new_empty(shape), context.positions.new_empty(shape))\n                        for _ in context.cross]\n        self.valid = torch.zeros(self.n, context.capacity, dtype=torch.bool, device=context.src.device)\n\n    @torch.inference_mode()\n    def select(self, index):\n        """Copy only initialized self-KV prefix, then continue at the SAME position."""\n        if index.ndim != 1 or not len(index):\n            raise ValueError(\'Nonempty 1-D active indices required\')\n        n, cap = len(index), self.c.capacity\n        self.allowed = self.allowed.index_select(0, index)\n        self.cross = [tuple(t.index_select(0, index) for t in pair) for pair in self.cross]\n        # Release each old layer as soon as its replacement is ready. Holding both\n        # complete caches would make compaction itself an avoidable T4 OOM peak.\n        for number in range(len(self.self_kv)):\n            pair = self.self_kv[number]\n            new_pair = []\n            for old in pair:\n                new = old.new_empty((n, self.c.heads, cap, self.c.dim))\n                new[:, :, :self.t].copy_(old[:, :, :self.t].index_select(0, index))\n                new_pair.append(new)\n            self.self_kv[number] = tuple(new_pair)\n        self.n = n\n        self.valid = self.valid.index_select(0, index)\n\n\ndef race_pick(probabilities, noise):\n    """Independent Exp(1) race: argmax(p/E) has categorical probabilities p.\n\n    Uses exactly the same FP32 softmax as baseline; RNG consumption can change.\n    Avoids multinomial\'s probability validation/reductions. Noise storage is reused.\n    """\n    noise.exponential_()\n    return torch.div(probabilities, noise, out=noise).argmax(-1)\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, compute_logprob=True,\n           options=None, stats=None):\n    options = options or Options()\n    d = Decoder(context, k, options.share_cross)\n    device, n = context.src.device, d.n\n    token = torch.full((n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(n, dtype=torch.bool, device=device)\n    output = torch.full((n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums, lengths = torch.zeros(n, device=device), torch.zeros(n, dtype=torch.long, device=device)\n    final_sums, final_lengths = torch.zeros_like(sums), torch.zeros_like(lengths)\n    original = torch.arange(n, device=device)\n    compacted = False\n    noise = None\n    host = event = None\n    pending = False\n    async_eos = options.async_eos and device.type == \'cuda\'\n    if async_eos:\n        host = torch.empty(n, dtype=torch.bool, pin_memory=True)\n        event = torch.cuda.Event()\n    history = []\n    checks = 0\n\n    def flush():\n        final_sums.index_copy_(0, original, sums)\n        final_lengths.index_copy_(0, original, lengths)\n\n    def inspect(snapshot):\n        nonlocal token, done, sums, lengths, original, compacted, noise, host\n        if bool(snapshot.all()):\n            return True\n        if not options.compact_done:\n            return False\n        keep = (~snapshot).nonzero().flatten()  # CPU only: never CUDA nonzero/item.\n        if options.compact_done and len(keep) <= d.n*(1-options.compact_fraction):\n            flush()\n            index = keep.to(device)\n            before = d.n\n            d.select(index)\n            token, done = token[index], done[index]\n            sums, lengths, original = sums[index], lengths[index], original[index]\n            compacted, noise = True, None\n            history.append({\'step\': d.t, \'before\': before, \'after\': d.n})\n            if async_eos:\n                host = torch.empty(d.n, dtype=torch.bool, pin_memory=True)\n        return False\n\n    try:\n        for t in range(context.capacity):\n            if t % 16 == 0:\n                C.check_deadline(deadline)\n            if pending and event.query():\n                pending = False\n                if inspect(host):\n                    break\n            logits = d.step(token)\n            if temperature is not None or compute_logprob:\n                logits = logits.float()\n            logits[:, :2] = -torch.inf\n            if temperature is None:\n                pick = logits.argmax(-1)\n            else:\n                probabilities = F.softmax(logits / temperature, -1)\n                if options.race_sampler:\n                    if noise is None:\n                        noise = torch.empty_like(probabilities)\n                    pick = race_pick(probabilities, noise)\n                else:\n                    pick = torch.multinomial(probabilities, 1).squeeze(-1)\n            if compute_logprob:\n                lp = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n                sums.add_(torch.where(done, 0., lp))\n            if compute_logprob or not options.lean_greedy:\n                lengths.add_(~done)\n            token = pick.masked_fill(done, C.PAD)\n            if compacted:\n                output[:, t].index_copy_(0, original, token)\n            else:\n                output[:, t] = token\n            done |= token.eq(C.EOS)\n            if (t+1) % options.eos_every == 0:\n                if async_eos:\n                    if not pending:\n                        host.copy_(done, non_blocking=True)\n                        event.record()\n                        pending = True\n                        checks += 1\n                else:\n                    checks += 1\n                    if inspect(done.cpu()):\n                        break\n    finally:\n        # Also retain pinned storage on deadline/OOM exceptions until copy finishes.\n        if pending:\n            event.synchronize()\n    flush()\n    if stats is not None:\n        stats.update(steps=d.t, eos_checks=checks, compactions=history,\n                     async_eos=async_eos, rng_stream_may_change=options.race_sampler or options.compact_done)\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (final_sums / final_lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'shape_specs.py': '"""Same dense architecture/FFN rule as L1; shapes alone change."""\nSHAPES = [(6,6,512), (6,8,640), (6,6,768), (6,6,640), (12,6,640),\n          (10,4,768), (8,6,640), (6,10,512), (4,4,768),\n          (8,8,640), (8,4,768), (6,5,768), (7,7,640)]\n\n\ndef name(shape):\n    e, d, w = shape\n    return f\'e{e}-d{d}-w{w}\'\n\n\ndef spec(shape):\n    e, d, w = shape\n    # C=1368, D=1712: ceil(8*d/3) to a multiple of 8.\n    return dict(d_model=w, heads=w//64, encoder_layers=e, decoder_layers=d,\n                swiglu_dim=((8*w+23)//24)*8, vocab=8195, max_positions=2048, dropout=0.)\n\n\ndef parameter_formula(shape):\n    e, d, w = shape; f = spec(shape)[\'swiglu_dim\']\n    encoder = e*(4*w*w+3*w*f+9*w+2*f)+2*w\n    decoder = d*(8*w*w+3*w*f+15*w+2*f)+2*w\n    embeddings = (8195+2048)*w\n    head = 8195*(w+1)\n    return dict(encoder=encoder, decoder=decoder, embeddings=embeddings, head=head,\n                total=encoder+decoder+embeddings+head)\n', 'shape_bench.py': '"""One offline T4 run. Dummy optimizer updates are timings, not model training.\n\nFrozen v6 decoder is used verbatim; only EOS logits are replaced by a shared\nlength schedule. Every shape sees identical padding and forced completion times.\n"""\nimport csv\nimport gc\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\nimport final_fast as Fast\nimport final_speed as Speed\nfrom shape_specs import SHAPES, name, spec, parameter_formula\n\n\ndef sync():\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n\n\ndef clean():\n    gc.collect()\n    if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n\ndef peak():\n    return dict(peak_allocated_bytes=torch.cuda.max_memory_allocated(),\n                peak_reserved_bytes=torch.cuda.max_memory_reserved()) if torch.cuda.is_available() else {}\n\n\ndef reset_peak():\n    sync()\n    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()\n\n\ndef tokens(lengths, device):\n    n = len(lengths); width = max(lengths)\n    x = torch.randint(3,8195,(n,width),device=device)\n    mask = torch.arange(width,device=device)[None,:] >= torch.tensor(lengths,device=device)[:,None]\n    return x.masked_fill(mask,0)\n\n\ndef batch(lengths, device):\n    src = tokens([a for a,b in lengths],device)\n    labels = tokens([b for a,b in lengths],device)\n    for i,(_,b) in enumerate(lengths): labels[i,b-1] = C.EOS\n    tgt = torch.zeros_like(labels); tgt[:,0] = C.BOS; tgt[:,1:] = labels[:,:-1]\n    return src,tgt,labels\n\n\ndef parameter_counts(model):\n    out = {k:sum(p.numel() for p in module.parameters()) for k,module in\n           [(\'encoder\',model.transformer.encoder),(\'decoder\',model.transformer.decoder),\n            (\'head\',model.head)]}\n    out[\'embeddings\'] = model.tokens.weight.numel()+model.positions.weight.numel()\n    out[\'total\'] = sum(p.numel() for p in model.parameters())\n    assert out[\'total\'] == sum(out[k] for k in (\'encoder\',\'decoder\',\'head\',\'embeddings\'))\n    return out\n\n\ndef update(model, opt, scaler, lengths, micro, device, deadline=None):\n    opt.zero_grad(set_to_none=True)\n    denom = sum(b for a,b in lengths)\n    for lo in range(0,len(lengths),micro):\n        C.check_deadline(deadline)\n        src,tgt,labels = batch(lengths[lo:lo+micro],device)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type==\'cuda\'):\n            logits = model(src,tgt)\n            loss = F.cross_entropy(logits.flatten(0,1), labels.flatten(),ignore_index=0,reduction=\'sum\')/denom\n        scaler.scale(loss).backward()\n        del src,tgt,labels,logits,loss\n    scaler.unscale_(opt)\n    norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n    # A random-init overflow must not make an artificially cheap Adam timing.\n    if not bool(torch.isfinite(norm)): raise FloatingPointError(\'Nonfinite benchmark gradient\')\n    scaler.step(opt); scaler.update()\n\n\ndef train_bench(model, data, device, deadline):\n    initial = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n    attempts = []; micro = None\n    for candidate in (64,32,16,8,4,2,1):\n        C.check_deadline(deadline)\n        opt = scaler = None\n        try:\n            model.load_state_dict(initial); model.train(); clean(); reset_peak()\n            opt = torch.optim.AdamW(model.parameters(),lr=1e-6,weight_decay=.01,fused=device.type==\'cuda\')\n            scaler = torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1.)\n            started = time.perf_counter()\n            update(model,opt,scaler,data[\'probe_batch\'],candidate,device,deadline); sync()\n            passed = device.type!=\'cuda\' or torch.cuda.max_memory_reserved()<.93*torch.cuda.get_device_properties(0).total_memory\n            attempts.append(dict(micro=candidate,status=\'passed\' if passed else \'headroom\',\n                                 seconds=time.perf_counter()-started,**peak()))\n            if passed: micro=candidate; break\n        except torch.cuda.OutOfMemoryError:\n            attempts.append(dict(micro=candidate,status=\'OOM\'))\n        finally:\n            model.zero_grad(set_to_none=True); opt=None; scaler=None; clean()\n    if micro is None: raise RuntimeError(\'No safe training micro-batch\')\n    model.load_state_dict(initial)\n    opt = torch.optim.AdamW(model.parameters(),lr=1e-6,weight_decay=.01,fused=device.type==\'cuda\')\n    scaler = torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1.)\n    # Complete first update outside measurement initializes Adam state/backend.\n    update(model,opt,scaler,data[\'train_batches\'][0],micro,device,deadline); sync()\n    rows = []; reset_peak()\n    for lengths in data[\'train_batches\']:\n        C.check_deadline(deadline); sync(); started=time.perf_counter()\n        update(model,opt,scaler,lengths,micro,device,deadline); sync()\n        rows.append(dict(seconds=time.perf_counter()-started, pairs=len(lengths),\n                         source_tokens=sum(a for a,b in lengths),target_tokens=sum(b for a,b in lengths)))\n    result = dict(micro=micro,effective_batch=128,probe=attempts,batches=rows,\n                  pairs_per_s=sum(r[\'pairs\'] for r in rows)/sum(r[\'seconds\'] for r in rows),\n                  seconds_per_effective128=float(np.mean([r[\'seconds\'] for r in rows])),\n                  **peak(),timer=\'CPU length collation/random ID creation+H2D+AMP forward/backward+clip+fused AdamW; synced wall\')\n    model.zero_grad(set_to_none=True); opt=None; scaler=None; clean()\n    model.load_state_dict(initial)  # Every decode uses the initial random model.\n    return result\n\n\nclass ForcedDecoder(Speed.Decoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        super().__init__(context,k,share_cross)\n        self.ends=context.forced_lengths.repeat_interleave(k)\n\n    @torch.inference_mode()\n    def step(self, ids):\n        logits=super().step(ids)\n        # No natural EOS: at each row\'s predetermined last step EOS is certain.\n        logits[:,C.EOS]=-torch.inf\n        force=self.t>=self.ends\n        logits.masked_fill_(force[:,None],-torch.inf)\n        logits[:,C.EOS]=torch.where(force,torch.zeros_like(logits[:,C.EOS]),logits[:,C.EOS])\n        return logits\n\n    @torch.inference_mode()\n    def select(self,index):\n        super().select(index); self.ends=self.ends.index_select(0,index)\n\n\nOPTIONS=Speed.Options(share_cross=True,async_eos=True,eos_every=32,compact_done=True,\n                      race_sampler=True,lean_greedy=True)\n\n\n@torch.inference_mode()\ndef forced(model, lengths, k, temperature, device, deadline):\n    src=tokens([a for a,b in lengths],device)\n    context=Fast.Context(model,src,512)\n    context.forced_lengths=torch.tensor([b for a,b in lengths],device=device)\n    old=Speed.Decoder; Speed.Decoder=ForcedDecoder\n    try:\n        stats={}\n        ids,lp,_=Speed.decode(context,k=k,temperature=temperature,compute_logprob=temperature is not None,\n                              options=OPTIONS,deadline=deadline,stats=stats)\n        ends=context.forced_lengths.cpu().repeat_interleave(k).tolist()\n        flat=ids.flatten(0,1)\n        assert all(int(flat[i,b-1])==C.EOS and not bool((flat[i,:b-1]==C.EOS).any()) for i,b in enumerate(ends))\n        return stats\n    finally: Speed.Decoder=old\n\n\ndef generation_bench(model,data,device,deadline):\n    model.half().eval(); out={}\n    lengths=data[\'generation1024\']\n    # Short warmup includes both argmax and race/logprob kernels.\n    for temp in (None,.7): forced(model,[[a,min(b,32)] for a,b in lengths[:32]],1,temp,device,deadline)\n    for label,k,temp in [(\'greedy\',1,None),(\'sample\',4,.7)]:\n        # 1024 greedy questions; sampling 256 questions x4 lanes =1024 sequences.\n        chosen=lengths if k==1 else [lengths[i] for i in np.linspace(0,1023,256).round().astype(int)]\n        oom=[]; lanes=1024\n        while True:\n            try:\n                clean(); reset_peak(); sync(); start=time.perf_counter(); stats=[]\n                for lo in range(0,len(chosen),lanes//k):\n                    C.check_deadline(deadline)\n                    stats.append(forced(model,chosen[lo:lo+lanes//k],k,temp,device,deadline))\n                sync(); seconds=time.perf_counter()-start\n                out[label]=dict(seconds=seconds,sequences=1024,questions=len(chosen),K=k,\n                    seq_per_s=1024/seconds,lanes=lanes,OOM_attempts=oom,decode_stats=stats,**peak(),\n                    timer=\'source IDs/H2D+encoder+cross-KV+forced v6 decode+D2H; no program execution\')\n                break\n            except torch.cuda.OutOfMemoryError:\n                oom.append(lanes); clean(); lanes//=2\n                if lanes<64: raise\n    return out\n\n\n@torch.inference_mode()\ndef encoder_bench(model,data,device,deadline):\n    # Prepared IDs are outside this timer, matching encoder compute rather than tokenizer.\n    parts=[tokens(data[\'test1919_source_lengths\'][lo:lo+256],device) for lo in range(0,1919,256)]\n    model.eval()\n    for src in parts[:1]: model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(0))\n    rows=[]; reset_peak()\n    for _ in range(3):\n        C.check_deadline(deadline); sync(); start=time.perf_counter()\n        for src in parts: model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(0))\n        sync(); rows.append(time.perf_counter()-start)\n    return dict(seconds=float(np.median(rows)),repeats=rows,questions=1919,batch=256,**peak())\n\n\ndef write(output,result):\n    output.mkdir(parents=True,exist_ok=True)\n    tmp=output/\'summary.tmp\'; tmp.write_text(json.dumps(result,indent=2,allow_nan=False)); tmp.replace(output/\'summary.json\')\n    fields=[\'shape\',\'params\',\'pairs_per_s\',\'seconds_per_effective128\',\'micro\',\'greedy_seq_per_s\',\'sample_seq_per_s\',\'encoder1919_s\',\'status\']\n    with (output/\'timings.csv\').open(\'w\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields); writer.writeheader()\n        for row in result[\'rows\']:\n            tr=row.get(\'training\',{}); gen=row.get(\'generation\',{})\n            writer.writerow(dict(shape=row[\'shape\'],params=row[\'parameters\'][\'total\'],\n                pairs_per_s=tr.get(\'pairs_per_s\'),seconds_per_effective128=tr.get(\'seconds_per_effective128\'),\n                micro=tr.get(\'micro\'),greedy_seq_per_s=gen.get(\'greedy\',{}).get(\'seq_per_s\'),\n                sample_seq_per_s=gen.get(\'sample\',{}).get(\'seq_per_s\'),\n                encoder1919_s=row.get(\'encoder\',{}).get(\'seconds\'),status=row[\'status\']))\n\n\ndef run(data_path,output,start,smoke=False):\n    device=torch.device(\'cpu\' if smoke else \'cuda:0\')\n    torch.set_num_threads(1); torch.set_num_interop_threads(1)\n    if not smoke:\n        assert torch.cuda.is_available() and \'T4\' in torch.cuda.get_device_name(0)\n        assert torch.__version__==\'2.11.0+cu128\' and torch.version.cuda==\'12.8\'\n    data=json.loads(Path(data_path).read_text()); output=Path(output)\n    deadline=start+1100\n    result=dict(format=\'nlp-hw02-shape-bench-v1\',status=\'running\',rows=[],\n        device=str(device),torch=torch.__version__,cuda=torch.version.cuda,random_init=True,\n        no_weights_saved=True,data_provenance=data[\'provenance\'],session_limit_s=1200,\n        length_source=\'verified cache v2 train; joint L3 C-main test output/source lengths\',\n        module_sha256={n:hashlib.sha256((Path(__file__).parent/n).read_bytes()).hexdigest() for n in\n            [\'final_core.py\',\'final_fast.py\',\'final_decode.py\',\'final_speed.py\',\'shape_specs.py\',\'shape_bench.py\']},\n        lengths_payload_sha256=hashlib.sha256(json.dumps(data,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest(),\n        no_end_to_end_timer=True)\n    write(output,result)\n    for shape in SHAPES:\n        if time.monotonic()>deadline-60: break\n        row=dict(shape=name(shape),spec=spec(shape),parameters=parameter_formula(shape),status=\'running\')\n        result[\'rows\'].append(row); write(output,result); model=None\n        try:\n            torch.manual_seed(42)\n            model=C.SimpleTransformer(**spec(shape)).to(device)\n            with torch.no_grad(): model.tokens.weight.div_(math.sqrt(model.width))\n            assert parameter_counts(model)==row[\'parameters\']\n            row[\'training\']=train_bench(model,data,device,deadline)\n            row[\'generation\']=generation_bench(model,data,device,deadline)\n            row[\'encoder\']=encoder_bench(model,data,device,deadline)\n            row[\'status\']=\'complete\'\n        except Exception as error:\n            row[\'status\']=\'failed\'; row[\'error\']=type(error).__name__+\': \'+str(error)[:300]\n        finally:\n            model=None; clean(); result[\'wall_s\']=time.monotonic()-start; write(output,result)\n        print(row[\'shape\'],row[\'status\'],row.get(\'training\',{}).get(\'pairs_per_s\'),flush=True)\n    result[\'status\']=\'complete\' if len(result[\'rows\'])==len(SHAPES) and all(r[\'status\']==\'complete\' for r in result[\'rows\']) else \'partial\'\n    result[\'wall_s\']=time.monotonic()-start; write(output,result)\n    return result\n'}
LENGTHS={'format': 'nlp-hw02-shape-lengths-v1', 'seed': 2026101507, 'cache_manifest_sha256': '46ef881bba60dba2563a25a7490becf05104a0537b70c3510629bef8c93902be', 'cache_rows': 9725737, 'source_tokens_per_epoch': 494009513, 'target_tokens_with_EOS_per_epoch': 540535346, 'mean_pair_tokens': 106.37187279483292, 'mean_target_tokens': 55.57782880618713, 'C_main_curve_tokens': {'24270': {'source': 157779528, 'target': 172672643, 'pair': 330452171}, '78056': {'source': 507472841, 'target': 555261630, 'pair': 1062734471}, '107090': {'source': 696263187, 'target': 761869798, 'pair': 1458132985}, '126452': {'source': 822143500, 'target': 899546043, 'pair': 1721689543}, '169956': {'source': 1104982191, 'target': 1209087303, 'pair': 2314069494}, '213386': {'source': 1387385553, 'target': 1518038882, 'pair': 2905424435}}, 'ordinary_order_sha256': '09cf6db8549b7df614fa71516553d23d4cbb3ca97c7911db0c6d599a8277ab7d', 'stress_curve_tokens': {'deep': {'main_target': 2071543365, 'main_pair': 3896075880, 'cooldown_target': 16478381, 'cooldown_pair': 30886269}, 'control': {'main_target': 2284396123, 'main_pair': 4295061244, 'cooldown_target': 23931327, 'cooldown_pair': 44837015}, 'L3-long': {'main_target': 1604893621, 'main_pair': 3021422708, 'cooldown_target': 59341237, 'cooldown_pair': 111212903}, 'P1-A': {'main_target': 1604893621, 'main_pair': 3021422708, 'cooldown_target': 245866689, 'cooldown_pair': 460800682}, 'P1-B': {'main_target': 1604893621, 'main_pair': 3021422708, 'cooldown_target': 245866689, 'cooldown_pair': 460800682}}, 'weighted_order_sha256': '72bc3dfd3a3e738979c489c0cf8f41cde092c41d4ae90f2de8378a034603a4db', 'token_accounting': 'presented tokens, not q10%-retained loss tokens; main excludes disposable cooldowns', 'train_batches': [[[22, 34], [25, 31], [25, 31], [26, 30], [26, 30], [27, 29], [27, 29], [28, 28], [29, 27], [30, 26], [30, 26], [31, 25], [32, 24], [33, 23], [34, 22], [37, 19], [23, 34], [23, 34], [23, 34], [23, 34], [26, 31], [26, 31], [27, 30], [27, 30], [27, 30], [27, 30], [28, 29], [28, 29], [28, 29], [28, 29], [28, 29], [28, 29], [28, 29], [28, 29], [28, 29], [30, 27], [30, 27], [30, 27], [30, 27], [31, 26], [31, 26], [31, 26], [31, 26], [31, 26], [32, 25], [32, 25], [33, 24], [33, 24], [34, 23], [25, 33], [26, 32], [27, 31], [27, 31], [27, 31], [28, 30], [28, 30], [28, 30], [28, 30], [28, 30], [29, 29], [29, 29], [29, 29], [29, 29], [30, 28], [30, 28], [30, 28], [30, 28], [31, 27], [31, 27], [31, 27], [31, 27], [32, 26], [32, 26], [32, 26], [33, 25], [33, 25], [33, 25], [34, 24], [35, 23], [35, 23], [21, 38], [23, 36], [23, 36], [23, 36], [24, 35], [26, 33], [27, 32], [27, 32], [27, 32], [27, 32], [28, 31], [28, 31], [28, 31], [28, 31], [28, 31], [29, 30], [29, 30], [30, 29], [30, 29], [31, 28], [31, 28], [31, 28], [31, 28], [31, 28], [32, 27], [32, 27], [32, 27], [33, 26], [33, 26], [33, 26], [25, 35], [26, 34], [26, 34], [27, 33], [27, 33], [27, 33], [28, 32], [28, 32], [28, 32], [28, 32], [28, 32], [28, 32], [28, 32], [29, 31], [29, 31], [29, 31], [29, 31], [29, 31]], [[27, 42], [27, 42], [27, 42], [28, 41], [28, 41], [30, 39], [30, 39], [31, 38], [31, 38], [31, 38], [32, 37], [32, 37], [32, 37], [32, 37], [33, 36], [33, 36], [34, 35], [34, 35], [34, 35], [34, 35], [34, 35], [35, 34], [35, 34], [36, 33], [36, 33], [36, 33], [36, 33], [37, 32], [37, 32], [37, 32], [37, 32], [38, 31], [38, 31], [39, 30], [39, 30], [39, 30], [40, 29], [40, 29], [40, 29], [41, 28], [42, 27], [42, 27], [42, 27], [45, 24], [45, 24], [46, 23], [22, 48], [24, 46], [27, 43], [30, 40], [30, 40], [30, 40], [31, 39], [31, 39], [31, 39], [31, 39], [31, 39], [32, 38], [32, 38], [33, 37], [33, 37], [33, 37], [33, 37], [33, 37], [33, 37], [34, 36], [34, 36], [35, 35], [35, 35], [36, 34], [36, 34], [36, 34], [36, 34], [36, 34], [36, 34], [36, 34], [37, 33], [37, 33], [37, 33], [37, 33], [37, 33], [37, 33], [37, 33], [38, 32], [38, 32], [38, 32], [40, 30], [41, 29], [41, 29], [41, 29], [42, 28], [42, 28], [44, 26], [47, 23], [25, 46], [26, 45], [28, 43], [30, 41], [31, 40], [32, 39], [32, 39], [32, 39], [33, 38], [33, 38], [33, 38], [33, 38], [34, 37], [34, 37], [34, 37], [34, 37], [34, 37], [35, 36], [36, 35], [36, 35], [36, 35], [36, 35], [36, 35], [36, 35], [36, 35], [36, 35], [36, 35], [37, 34], [37, 34], [37, 34], [37, 34], [37, 34], [37, 34], [38, 33]], [[39, 40], [39, 40], [40, 39], [40, 39], [40, 39], [40, 39], [41, 38], [41, 38], [41, 38], [41, 38], [41, 38], [41, 38], [42, 37], [42, 37], [42, 37], [43, 36], [44, 35], [44, 35], [44, 35], [46, 33], [46, 33], [46, 33], [47, 32], [47, 32], [47, 32], [47, 32], [47, 32], [48, 31], [50, 29], [57, 22], [59, 20], [30, 50], [30, 50], [30, 50], [31, 49], [32, 48], [33, 47], [33, 47], [33, 47], [33, 47], [33, 47], [33, 47], [33, 47], [34, 46], [34, 46], [35, 45], [36, 44], [36, 44], [36, 44], [36, 44], [37, 43], [38, 42], [38, 42], [39, 41], [39, 41], [39, 41], [39, 41], [41, 39], [41, 39], [42, 38], [42, 38], [42, 38], [42, 38], [42, 38], [44, 36], [45, 35], [45, 35], [46, 34], [46, 34], [47, 33], [48, 32], [48, 32], [48, 32], [52, 28], [55, 25], [30, 51], [32, 49], [32, 49], [32, 49], [33, 48], [33, 48], [33, 48], [33, 48], [34, 47], [34, 47], [35, 46], [35, 46], [36, 45], [36, 45], [36, 45], [37, 44], [37, 44], [37, 44], [37, 44], [38, 43], [38, 43], [38, 43], [38, 43], [38, 43], [38, 43], [39, 42], [39, 42], [39, 42], [39, 42], [39, 42], [40, 41], [40, 41], [40, 41], [40, 41], [40, 41], [40, 41], [41, 40], [41, 40], [41, 40], [41, 40], [42, 39], [42, 39], [44, 37], [44, 37], [44, 37], [45, 36], [45, 36], [45, 36], [45, 36], [46, 35], [47, 34], [47, 34], [48, 33]], [[41, 49], [42, 48], [42, 48], [42, 48], [42, 48], [44, 46], [46, 44], [46, 44], [46, 44], [46, 44], [48, 42], [48, 42], [49, 41], [50, 40], [50, 40], [50, 40], [51, 39], [51, 39], [52, 38], [53, 37], [54, 36], [55, 35], [31, 60], [35, 56], [36, 55], [36, 55], [37, 54], [37, 54], [38, 53], [38, 53], [39, 52], [40, 51], [40, 51], [41, 50], [41, 50], [41, 50], [41, 50], [42, 49], [42, 49], [43, 48], [43, 48], [43, 48], [43, 48], [44, 47], [44, 47], [44, 47], [44, 47], [45, 46], [47, 44], [48, 43], [48, 43], [49, 42], [49, 42], [49, 42], [49, 42], [49, 42], [50, 41], [50, 41], [51, 40], [51, 40], [52, 39], [55, 36], [55, 36], [56, 35], [62, 29], [29, 63], [33, 59], [35, 57], [35, 57], [36, 56], [37, 55], [37, 55], [38, 54], [38, 54], [39, 53], [40, 52], [42, 50], [42, 50], [43, 49], [43, 49], [43, 49], [43, 49], [43, 49], [43, 49], [43, 49], [43, 49], [43, 49], [44, 48], [44, 48], [44, 48], [44, 48], [45, 47], [45, 47], [45, 47], [46, 46], [46, 46], [46, 46], [47, 45], [47, 45], [47, 45], [47, 45], [48, 44], [48, 44], [49, 43], [50, 42], [51, 41], [51, 41], [53, 39], [54, 38], [55, 37], [65, 27], [32, 61], [33, 60], [34, 59], [35, 58], [36, 57], [36, 57], [38, 55], [38, 55], [38, 55], [39, 54], [40, 53], [40, 53], [41, 52], [41, 52], [42, 51], [43, 50], [43, 50]], [[52, 50], [53, 49], [54, 48], [56, 46], [57, 45], [58, 44], [61, 41], [62, 40], [84, 18], [39, 64], [39, 64], [40, 63], [41, 62], [41, 62], [41, 62], [42, 61], [43, 60], [44, 59], [44, 59], [45, 58], [45, 58], [45, 58], [45, 58], [46, 57], [46, 57], [48, 55], [48, 55], [48, 55], [48, 55], [50, 53], [51, 52], [51, 52], [53, 50], [54, 49], [54, 49], [54, 49], [54, 49], [54, 49], [55, 48], [55, 48], [55, 48], [56, 47], [56, 47], [57, 46], [57, 46], [58, 45], [59, 44], [61, 42], [61, 42], [62, 41], [38, 66], [38, 66], [40, 64], [42, 62], [43, 61], [44, 60], [44, 60], [44, 60], [44, 60], [44, 60], [46, 58], [46, 58], [47, 57], [47, 57], [48, 56], [48, 56], [48, 56], [49, 55], [49, 55], [49, 55], [50, 54], [50, 54], [51, 53], [51, 53], [51, 53], [51, 53], [52, 52], [52, 52], [52, 52], [54, 50], [59, 45], [59, 45], [60, 44], [61, 43], [64, 40], [68, 36], [69, 35], [38, 67], [40, 65], [40, 65], [41, 64], [42, 63], [42, 63], [43, 62], [44, 61], [44, 61], [45, 60], [45, 60], [46, 59], [48, 57], [48, 57], [48, 57], [49, 56], [49, 56], [49, 56], [50, 55], [50, 55], [51, 54], [52, 53], [52, 53], [52, 53], [52, 53], [54, 51], [54, 51], [54, 51], [55, 50], [55, 50], [57, 48], [57, 48], [58, 47], [60, 45], [61, 44], [65, 40], [66, 39], [68, 37], [79, 26], [31, 75], [37, 69]], [[56, 60], [58, 58], [58, 58], [58, 58], [59, 57], [59, 57], [60, 56], [60, 56], [60, 56], [61, 55], [62, 54], [65, 51], [66, 50], [68, 48], [69, 47], [39, 78], [39, 78], [43, 74], [43, 74], [44, 73], [44, 73], [46, 71], [47, 70], [47, 70], [48, 69], [49, 68], [49, 68], [49, 68], [51, 66], [52, 65], [53, 64], [53, 64], [54, 63], [55, 62], [56, 61], [56, 61], [57, 60], [58, 59], [59, 58], [59, 58], [60, 57], [61, 56], [62, 55], [62, 55], [62, 55], [63, 54], [64, 53], [65, 52], [66, 51], [67, 50], [70, 47], [71, 46], [74, 43], [35, 83], [42, 76], [44, 74], [45, 73], [47, 71], [48, 70], [50, 68], [51, 67], [51, 67], [51, 67], [53, 65], [54, 64], [57, 61], [59, 59], [60, 58], [60, 58], [61, 57], [61, 57], [63, 55], [65, 53], [65, 53], [67, 51], [72, 46], [73, 45], [75, 43], [39, 80], [41, 78], [41, 78], [44, 75], [48, 71], [49, 70], [50, 69], [50, 69], [50, 69], [50, 69], [51, 68], [52, 67], [52, 67], [53, 66], [53, 66], [53, 66], [53, 66], [53, 66], [55, 64], [58, 61], [59, 60], [61, 58], [62, 57], [62, 57], [63, 56], [64, 55], [64, 55], [66, 53], [69, 50], [69, 50], [71, 48], [71, 48], [39, 81], [42, 78], [44, 76], [44, 76], [45, 75], [45, 75], [47, 73], [48, 72], [50, 70], [51, 69], [52, 68], [52, 68], [59, 61], [60, 60], [60, 60], [60, 60], [63, 57], [63, 57]], [[64, 70], [67, 67], [68, 66], [68, 66], [69, 65], [71, 63], [73, 61], [73, 61], [78, 56], [35, 100], [38, 97], [44, 91], [48, 87], [50, 85], [50, 85], [53, 82], [54, 81], [57, 78], [57, 78], [59, 76], [61, 74], [61, 74], [62, 73], [63, 72], [63, 72], [65, 70], [65, 70], [66, 69], [66, 69], [68, 67], [70, 65], [70, 65], [72, 63], [72, 63], [72, 63], [73, 62], [76, 59], [76, 59], [78, 57], [84, 51], [96, 39], [48, 88], [53, 83], [56, 80], [56, 80], [56, 80], [59, 77], [63, 73], [67, 69], [69, 67], [70, 66], [76, 60], [76, 60], [77, 59], [77, 59], [78, 58], [83, 53], [88, 48], [50, 87], [51, 86], [51, 86], [53, 84], [55, 82], [56, 81], [58, 79], [59, 78], [60, 77], [64, 73], [65, 72], [67, 70], [75, 62], [76, 61], [79, 58], [80, 57], [80, 57], [87, 50], [43, 95], [47, 91], [50, 88], [56, 82], [60, 78], [60, 78], [63, 75], [65, 73], [66, 72], [66, 72], [66, 72], [70, 68], [71, 67], [72, 66], [74, 64], [75, 63], [77, 61], [90, 48], [100, 38], [47, 92], [52, 87], [55, 84], [56, 83], [56, 83], [56, 83], [57, 82], [57, 82], [57, 82], [58, 81], [60, 79], [62, 77], [63, 76], [64, 75], [67, 72], [67, 72], [67, 72], [69, 70], [72, 67], [73, 66], [75, 64], [76, 63], [76, 63], [82, 57], [82, 57], [84, 55], [52, 88], [54, 86], [56, 84], [58, 82], [58, 82], [59, 81], [59, 81]], [[81, 79], [85, 75], [85, 75], [90, 70], [126, 34], [43, 118], [60, 101], [62, 99], [63, 98], [67, 94], [71, 90], [74, 87], [74, 87], [75, 86], [76, 85], [83, 78], [90, 71], [63, 99], [64, 98], [66, 96], [67, 95], [68, 94], [69, 93], [73, 89], [77, 85], [78, 84], [82, 80], [87, 75], [87, 75], [94, 68], [38, 125], [52, 111], [59, 104], [60, 103], [74, 89], [76, 87], [77, 86], [102, 61], [126, 37], [43, 121], [50, 114], [56, 108], [59, 105], [60, 104], [66, 98], [78, 86], [79, 85], [80, 84], [81, 83], [81, 83], [82, 82], [83, 81], [89, 75], [95, 69], [95, 69], [98, 66], [54, 111], [62, 103], [66, 99], [67, 98], [68, 97], [71, 94], [77, 88], [88, 77], [70, 96], [70, 96], [75, 91], [75, 91], [76, 90], [79, 87], [84, 82], [84, 82], [85, 81], [86, 80], [86, 80], [86, 80], [91, 75], [93, 73], [98, 68], [60, 107], [62, 105], [67, 100], [77, 90], [79, 88], [80, 87], [58, 110], [70, 98], [75, 93], [78, 90], [82, 86], [82, 86], [85, 83], [90, 78], [92, 76], [94, 74], [108, 60], [67, 102], [70, 99], [71, 98], [77, 92], [83, 86], [58, 112], [73, 97], [73, 97], [74, 96], [76, 94], [77, 93], [78, 92], [79, 91], [92, 78], [100, 70], [100, 70], [101, 69], [55, 116], [60, 111], [65, 106], [68, 103], [73, 98], [74, 97], [75, 96], [81, 90], [81, 90], [82, 89], [87, 84], [95, 76], [101, 70], [51, 121], [56, 116]]], 'probe_batch': [[234, 39], [74, 380], [170, 156], [137, 190], [132, 195], [164, 163], [150, 177], [68, 260], [132, 196], [146, 182], [48, 280], [152, 176], [92, 237], [158, 171], [148, 181], [74, 255], [177, 153], [125, 205], [169, 161], [155, 175], [93, 238], [136, 195], [122, 209], [140, 191], [84, 248], [134, 198], [166, 166], [166, 166], [139, 193], [137, 195], [130, 203], [82, 251], [119, 214], [97, 236], [175, 158], [131, 202], [166, 168], [153, 181], [152, 182], [133, 201], [138, 196], [143, 192], [112, 223], [143, 192], [54, 281], [156, 179], [143, 192], [110, 226], [68, 268], [142, 194], [105, 231], [135, 201], [89, 247], [67, 269], [115, 221], [153, 184], [152, 185], [111, 226], [142, 195], [152, 185], [123, 214], [135, 202], [40, 297], [128, 210], [90, 248], [129, 210], [141, 199], [104, 236], [193, 148], [151, 190], [114, 227], [142, 199], [127, 215], [67, 275], [61, 282], [162, 182], [83, 262], [92, 254], [150, 196], [165, 182], [141, 207], [74, 274], [87, 261], [167, 182], [52, 297], [193, 157], [160, 190], [150, 200], [149, 201], [79, 272], [153, 198], [148, 203], [137, 215], [75, 277], [137, 215], [137, 215], [72, 283], [160, 197], [85, 273], [78, 280], [163, 198], [62, 300], [144, 218], [144, 218], [62, 300], [62, 300], [83, 280], [101, 263], [140, 224], [99, 269], [51, 324], [66, 309], [139, 238], [71, 306], [197, 180], [111, 268], [153, 226], [87, 294], [84, 305], [79, 325], [66, 339], [84, 329], [59, 359], [120, 298], [75, 350], [141, 291], [226, 216], [74, 380]], 'test1919_source_lengths': [8, 15, 18, 18, 18, 18, 18, 18, 19, 19, 20, 20, 20, 20, 20, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 35, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 36, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 43, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 47, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 53, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 55, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 56, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 59, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 62, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 67, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 69, 69, 69, 69, 69, 69, 69, 69, 69, 69, 69, 69, 69, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 70, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 71, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 72, 73, 73, 73, 73, 73, 73, 73, 73, 73, 73, 73, 74, 74, 74, 74, 74, 74, 74, 74, 74, 74, 74, 75, 75, 75, 75, 75, 75, 75, 75, 76, 76, 76, 76, 76, 76, 76, 76, 76, 77, 77, 77, 77, 77, 77, 77, 77, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 81, 81, 81, 81, 81, 81, 81, 81, 81, 81, 82, 82, 82, 82, 82, 82, 83, 83, 83, 83, 83, 83, 83, 84, 84, 84, 84, 84, 84, 84, 84, 85, 85, 85, 85, 85, 85, 85, 85, 85, 86, 86, 86, 86, 86, 86, 86, 86, 87, 87, 87, 87, 87, 87, 87, 88, 88, 88, 88, 89, 89, 89, 89, 89, 90, 90, 90, 90, 90, 91, 91, 93, 93, 93, 93, 94, 94, 94, 94, 94, 94, 94, 95, 95, 96, 96, 98, 98, 98, 98, 98, 99, 99, 99, 99, 101, 101, 101, 103, 103, 103, 104, 104, 105, 107, 107, 107, 107, 108, 109, 109, 111, 114, 115, 116, 117, 117, 118, 120, 122, 123, 124, 126, 126, 128, 131, 133, 139, 145, 147, 159, 169, 203], 'generation1024': [[8, 58], [18, 21], [18, 30], [18, 33], [18, 26], [19, 32], [20, 35], [20, 24], [21, 28], [21, 30], [21, 22], [22, 27], [22, 48], [22, 47], [22, 32], [22, 55], [23, 30], [23, 22], [23, 34], [23, 21], [23, 53], [23, 25], [23, 18], [24, 38], [24, 26], [24, 24], [24, 19], [24, 22], [24, 31], [24, 32], [24, 27], [24, 30], [24, 44], [25, 29], [25, 33], [25, 23], [25, 31], [25, 17], [25, 33], [25, 32], [25, 22], [25, 22], [25, 30], [25, 42], [25, 36], [25, 30], [25, 36], [26, 33], [26, 26], [26, 25], [26, 28], [26, 29], [26, 33], [26, 28], [27, 34], [27, 45], [27, 32], [27, 25], [27, 41], [27, 26], [27, 28], [27, 42], [27, 54], [27, 28], [27, 38], [27, 28], [27, 34], [28, 18], [28, 30], [28, 29], [28, 31], [28, 35], [28, 28], [28, 33], [28, 34], [28, 25], [28, 35], [28, 37], [28, 31], [28, 35], [28, 29], [28, 31], [28, 39], [28, 43], [28, 31], [28, 31], [28, 29], [28, 39], [28, 44], [28, 30], [28, 33], [28, 48], [28, 24], [29, 37], [29, 28], [29, 43], [29, 27], [29, 29], [29, 32], [29, 35], [29, 31], [29, 24], [29, 27], [29, 34], [29, 30], [29, 32], [29, 34], [29, 25], [29, 50], [29, 56], [29, 33], [29, 37], [29, 30], [30, 36], [30, 27], [30, 25], [30, 39], [30, 34], [30, 25], [30, 33], [30, 45], [30, 43], [30, 28], [30, 26], [30, 34], [30, 34], [30, 38], [30, 37], [30, 53], [30, 43], [30, 25], [30, 34], [31, 60], [31, 33], [31, 33], [31, 50], [31, 38], [31, 26], [31, 35], [31, 41], [31, 33], [31, 49], [31, 35], [31, 26], [31, 51], [31, 25], [31, 43], [31, 33], [31, 26], [31, 59], [31, 27], [31, 53], [31, 25], [31, 26], [32, 40], [32, 22], [32, 76], [32, 29], [32, 19], [32, 36], [32, 45], [32, 30], [32, 35], [32, 36], [32, 43], [32, 38], [32, 42], [32, 35], [32, 40], [32, 39], [32, 36], [32, 29], [32, 33], [32, 30], [32, 38], [32, 40], [32, 43], [32, 35], [33, 17], [33, 44], [33, 32], [33, 33], [33, 38], [33, 34], [33, 36], [33, 34], [33, 37], [33, 44], [33, 32], [33, 49], [33, 32], [33, 31], [33, 30], [33, 33], [33, 37], [33, 28], [33, 59], [33, 50], [33, 31], [33, 49], [33, 37], [33, 27], [34, 22], [34, 29], [34, 40], [34, 40], [34, 30], [34, 33], [34, 34], [34, 60], [34, 29], [34, 31], [34, 36], [34, 56], [34, 44], [34, 49], [34, 52], [34, 24], [34, 35], [34, 63], [34, 37], [34, 35], [34, 23], [34, 53], [34, 31], [34, 54], [34, 25], [34, 48], [35, 39], [35, 35], [35, 36], [35, 30], [35, 36], [35, 64], [35, 46], [35, 58], [35, 49], [35, 78], [35, 41], [35, 37], [35, 35], [35, 28], [35, 52], [35, 30], [35, 32], [35, 24], [35, 39], [35, 36], [35, 38], [35, 30], [35, 64], [35, 31], [35, 52], [35, 50], [35, 42], [35, 52], [35, 32], [35, 35], [35, 33], [36, 26], [36, 45], [36, 26], [36, 22], [36, 34], [36, 43], [36, 57], [36, 28], [36, 39], [36, 31], [36, 43], [36, 37], [36, 34], [36, 32], [36, 43], [36, 28], [36, 31], [36, 31], [36, 49], [36, 28], [36, 79], [36, 37], [36, 33], [36, 67], [36, 34], [37, 47], [37, 27], [37, 54], [37, 59], [37, 46], [37, 36], [37, 35], [37, 37], [37, 44], [37, 30], [37, 27], [37, 48], [37, 24], [37, 72], [37, 32], [37, 43], [37, 34], [37, 28], [37, 28], [37, 61], [37, 38], [37, 44], [37, 29], [37, 35], [37, 32], [38, 44], [38, 42], [38, 31], [38, 34], [38, 33], [38, 32], [38, 26], [38, 25], [38, 62], [38, 35], [38, 56], [38, 38], [38, 35], [38, 34], [38, 26], [38, 37], [38, 33], [38, 42], [38, 27], [38, 36], [38, 34], [38, 32], [38, 64], [38, 34], [38, 40], [38, 51], [39, 43], [39, 46], [39, 52], [39, 50], [39, 32], [39, 42], [39, 47], [39, 26], [39, 29], [39, 32], [39, 52], [39, 38], [39, 84], [39, 31], [39, 63], [39, 79], [39, 32], [39, 40], [39, 29], [39, 52], [39, 34], [39, 53], [39, 90], [39, 46], [39, 25], [39, 47], [39, 42], [40, 55], [40, 48], [40, 24], [40, 62], [40, 31], [40, 49], [40, 57], [40, 88], [40, 66], [40, 53], [40, 40], [40, 38], [40, 48], [40, 62], [40, 57], [40, 46], [40, 32], [40, 66], [40, 38], [40, 38], [40, 42], [40, 45], [40, 24], [40, 45], [40, 37], [40, 34], [40, 52], [41, 49], [41, 44], [41, 45], [41, 48], [41, 45], [41, 52], [41, 32], [41, 33], [41, 60], [41, 38], [41, 59], [41, 36], [41, 30], [41, 41], [41, 94], [41, 33], [41, 33], [41, 43], [41, 53], [41, 32], [41, 38], [41, 51], [41, 33], [41, 43], [41, 35], [41, 49], [42, 37], [42, 49], [42, 51], [42, 61], [42, 39], [42, 33], [42, 36], [42, 27], [42, 47], [42, 57], [42, 45], [42, 55], [42, 84], [42, 36], [42, 39], [42, 41], [42, 33], [42, 39], [42, 34], [42, 64], [42, 45], [42, 25], [42, 50], [42, 38], [42, 63], [43, 43], [43, 32], [43, 51], [43, 44], [43, 51], [43, 44], [43, 36], [43, 46], [43, 39], [43, 53], [43, 52], [43, 55], [43, 29], [43, 55], [43, 53], [43, 60], [43, 52], [43, 31], [43, 17], [43, 50], [43, 47], [43, 41], [43, 89], [43, 47], [43, 22], [43, 30], [43, 57], [43, 45], [43, 34], [43, 56], [44, 39], [44, 44], [44, 29], [44, 38], [44, 47], [44, 62], [44, 27], [44, 44], [44, 46], [44, 44], [44, 40], [44, 37], [44, 54], [44, 48], [44, 33], [44, 49], [44, 33], [44, 78], [44, 50], [44, 94], [44, 47], [45, 61], [45, 58], [45, 54], [45, 41], [45, 31], [45, 52], [45, 35], [45, 28], [45, 53], [45, 87], [45, 56], [45, 39], [45, 28], [45, 37], [45, 69], [45, 66], [45, 34], [45, 45], [45, 60], [45, 74], [45, 44], [45, 51], [46, 73], [46, 66], [46, 44], [46, 43], [46, 48], [46, 32], [46, 50], [46, 65], [46, 71], [46, 32], [46, 45], [46, 50], [46, 32], [46, 27], [46, 59], [46, 47], [46, 57], [46, 34], [46, 44], [46, 47], [46, 39], [46, 36], [46, 31], [46, 33], [47, 89], [47, 53], [47, 60], [47, 49], [47, 38], [47, 35], [47, 37], [47, 42], [47, 71], [47, 44], [47, 36], [47, 53], [47, 39], [47, 36], [47, 75], [47, 90], [47, 39], [47, 44], [48, 43], [48, 55], [48, 75], [48, 98], [48, 48], [48, 41], [48, 33], [48, 50], [48, 78], [48, 33], [48, 44], [48, 51], [48, 63], [48, 37], [48, 44], [48, 60], [48, 46], [48, 57], [48, 37], [48, 42], [48, 55], [48, 61], [49, 37], [49, 50], [49, 49], [49, 66], [49, 63], [49, 68], [49, 51], [49, 32], [49, 60], [49, 48], [49, 81], [49, 54], [49, 45], [49, 68], [49, 48], [49, 45], [49, 41], [49, 46], [49, 45], [49, 59], [49, 33], [50, 53], [50, 52], [50, 43], [50, 47], [50, 70], [50, 50], [50, 40], [50, 24], [50, 62], [50, 47], [50, 65], [50, 51], [50, 32], [50, 53], [50, 50], [50, 38], [50, 52], [51, 47], [51, 39], [51, 56], [51, 63], [51, 58], [51, 34], [51, 87], [51, 73], [51, 66], [51, 72], [51, 49], [51, 102], [51, 78], [51, 55], [51, 42], [51, 54], [51, 49], [51, 36], [51, 51], [52, 65], [52, 51], [52, 70], [52, 58], [52, 44], [52, 54], [52, 67], [52, 28], [52, 44], [52, 113], [52, 36], [52, 82], [52, 55], [52, 113], [52, 44], [52, 67], [52, 45], [52, 79], [52, 76], [52, 47], [52, 36], [52, 43], [52, 63], [53, 56], [53, 39], [53, 27], [53, 49], [53, 48], [53, 78], [53, 71], [53, 56], [53, 72], [53, 53], [53, 43], [53, 63], [53, 47], [53, 34], [53, 43], [53, 48], [53, 42], [53, 44], [53, 63], [53, 86], [53, 70], [53, 60], [53, 54], [54, 59], [54, 61], [54, 54], [54, 54], [54, 39], [54, 86], [54, 41], [54, 33], [54, 76], [54, 79], [54, 61], [54, 47], [54, 91], [55, 54], [55, 61], [55, 74], [55, 62], [55, 60], [55, 40], [55, 57], [55, 25], [55, 27], [55, 34], [55, 64], [55, 64], [55, 48], [55, 70], [55, 29], [55, 48], [55, 68], [55, 79], [56, 35], [56, 49], [56, 64], [56, 25], [56, 46], [56, 47], [56, 39], [56, 61], [56, 67], [56, 39], [56, 78], [56, 46], [56, 49], [56, 48], [56, 70], [56, 31], [57, 37], [57, 82], [57, 109], [57, 66], [57, 41], [57, 57], [57, 63], [57, 82], [57, 49], [57, 84], [57, 19], [57, 54], [57, 79], [57, 66], [57, 110], [57, 51], [57, 86], [58, 69], [58, 43], [58, 88], [58, 74], [58, 76], [58, 63], [58, 50], [58, 80], [58, 107], [58, 59], [58, 64], [59, 50], [59, 73], [59, 55], [59, 76], [59, 59], [59, 42], [59, 65], [59, 73], [59, 38], [59, 71], [59, 60], [59, 69], [59, 38], [60, 86], [60, 38], [60, 57], [60, 51], [60, 38], [60, 94], [60, 76], [60, 64], [60, 61], [60, 74], [60, 34], [60, 51], [61, 72], [61, 71], [61, 48], [61, 29], [61, 64], [61, 44], [61, 49], [61, 56], [61, 34], [62, 58], [62, 114], [62, 56], [62, 63], [62, 57], [62, 61], [62, 40], [62, 62], [62, 79], [62, 50], [63, 64], [63, 62], [63, 81], [63, 46], [63, 69], [63, 66], [63, 53], [63, 42], [63, 82], [64, 50], [64, 86], [64, 85], [64, 75], [64, 45], [64, 50], [64, 72], [64, 72], [64, 33], [64, 43], [64, 72], [64, 41], [64, 61], [64, 59], [64, 53], [65, 71], [65, 40], [65, 49], [65, 42], [65, 60], [65, 66], [65, 103], [65, 50], [65, 35], [65, 49], [65, 56], [65, 17], [65, 57], [66, 37], [66, 47], [66, 50], [66, 77], [66, 40], [66, 55], [66, 82], [66, 71], [66, 44], [66, 87], [66, 58], [66, 81], [67, 49], [67, 84], [67, 68], [67, 37], [67, 84], [67, 52], [67, 52], [67, 46], [67, 38], [67, 40], [67, 63], [67, 72], [67, 39], [67, 82], [68, 100], [68, 36], [68, 73], [68, 63], [68, 103], [68, 42], [68, 92], [68, 66], [68, 75], [68, 75], [68, 109], [68, 42], [69, 90], [69, 107], [69, 96], [69, 89], [69, 85], [69, 87], [69, 33], [70, 47], [70, 60], [70, 54], [70, 71], [70, 45], [70, 46], [70, 15], [70, 68], [71, 79], [71, 94], [71, 92], [71, 84], [71, 75], [71, 87], [71, 51], [71, 108], [71, 80], [71, 86], [72, 71], [72, 77], [72, 44], [72, 62], [72, 64], [72, 94], [72, 66], [72, 53], [72, 65], [73, 98], [73, 74], [73, 97], [73, 83], [73, 33], [73, 62], [74, 68], [74, 98], [74, 59], [74, 96], [74, 58], [74, 64], [75, 74], [75, 94], [75, 46], [75, 56], [76, 41], [76, 81], [76, 99], [76, 120], [76, 68], [77, 52], [77, 64], [77, 59], [77, 90], [78, 88], [78, 59], [78, 60], [78, 83], [78, 36], [78, 52], [79, 60], [79, 85], [79, 69], [79, 45], [79, 68], [79, 54], [79, 58], [80, 60], [80, 65], [80, 85], [80, 47], [80, 70], [80, 96], [80, 72], [81, 80], [81, 63], [81, 122], [81, 38], [81, 43], [82, 68], [82, 83], [82, 53], [83, 88], [83, 67], [83, 68], [83, 72], [84, 81], [84, 78], [84, 115], [84, 55], [85, 81], [85, 53], [85, 78], [85, 76], [85, 66], [86, 31], [86, 117], [86, 79], [86, 75], [87, 32], [87, 95], [87, 109], [87, 77], [88, 51], [88, 63], [89, 54], [89, 50], [89, 57], [90, 113], [90, 124], [90, 78], [91, 48], [93, 60], [93, 89], [94, 39], [94, 87], [94, 109], [94, 48], [95, 86], [96, 80], [98, 117], [98, 124], [99, 122], [99, 73], [101, 89], [101, 57], [103, 77], [103, 99], [104, 66], [107, 130], [107, 67], [108, 92], [109, 109], [111, 79], [115, 95], [117, 112], [118, 83], [122, 126], [124, 89], [126, 147], [131, 86], [133, 91], [145, 148], [159, 116], [203, 84]], 'provenance': {'offset_sha256': {'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00000/source-offsets.npy': 'a6b8ae1fc58f654d9f3fe62caea376b27072512ae291c5cd25528052540ef6f4', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00000/target-offsets.npy': 'd60baf1f54203c9bc1aa5df2ae8891f2051b6f0bc73d5f019827c1d6e22aa03b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00001/source-offsets.npy': '4aa2468a576edc6c64389ac9784a2640cff1d388b00ab3e39361046da31327c6', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00001/target-offsets.npy': '2bb86ee1fc9b2640a84165764badd7f84586167e694dc77d21e8ef44f79659f5', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00002/source-offsets.npy': 'ca1d89e63db5ebe8b3cd8b4f71f27d2e70b2380e07103fd7a8c0c502fc8d99c9', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00002/target-offsets.npy': 'e953f07974ae969f666f9983202bd4a18b27f25979f47cab9ab7ca2d33c9ae32', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00003/source-offsets.npy': '274f3680b0657a3fc8d551f6eeb43cbd09ab7be562158bd8096eb252d1704b2b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00003/target-offsets.npy': '98eb35f3f14fa0d77600d4d4e95e33163e9afa55a1200dcea60732e13122ef25', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00004/source-offsets.npy': '6163a7bc5265333c99c985246b3ee9f07b26c57b624b5cd7baab666d4a0147bf', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00004/target-offsets.npy': '6ea7e637d45be93526a643f288f47da0a440d21fdd506ae31edeca69fa677a2f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00005/source-offsets.npy': '4ce74d4c1481f1a150eef913a8d96f6083f57cd369e526f0ca926ac4df94021e', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00005/target-offsets.npy': 'dd33fddf5e37e89b67ad52321e7ca7d25e39dfe1e14b4ec2b58a534f0246db41', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00006/source-offsets.npy': '1dfa56e4b3f2f114526563152afbc6998c569fd625c8d971fa1a31e2f351c5ae', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00006/target-offsets.npy': '09549319fc6e11afa7956d96d9083367da74cf7f0414884201fb5958d16bda1c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00007/source-offsets.npy': 'c071a7e1ab9c4de7d23b74051a0010ad2156b40d0831048731778dfec1575e4a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00007/target-offsets.npy': '8daa40ed07693eaf94be94d56b49dc11c62e7e3215fb3fb8cd3a5cd9b5fc925c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00008/source-offsets.npy': '92b172f65853a45fbb005bb477b9e84803baf6352e73428ed8a5edd756b761be', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00008/target-offsets.npy': 'd7974a4b376f99bd69bb903d5c106f3622ab0681f37b400bb25f348ce6285c60', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00009/source-offsets.npy': '3bbd760de9e090de68672cedd3bcec68254a4d4346a80463526d49009f53830f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00009/target-offsets.npy': 'a6d9f83113ef83e4c23fd0d398bfe9140c53b07aa029a01234789bdb66d4ea09', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00010/source-offsets.npy': '8153015e36eec83b6297674f13e16565645855a7e321468c772181705c0707d7', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00010/target-offsets.npy': 'a200b2678f574bbe423ea80168bf85e9ac90a5b897aa086374102fd61cb870b1', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00011/source-offsets.npy': '9179b13ff0fadd1f9c4aeac3241fb07b35648bdcf90783e8ad7a895cb9b95506', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00011/target-offsets.npy': 'e89d680d53e86d3270ecf55b2a7de8668376fc380a5ca769b0011d2ca4c1b07c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00012/source-offsets.npy': '47d8d2ffc856cc5d284b852899542bca6e5ddb3076cc609f3416a358a2a05fd8', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00012/target-offsets.npy': 'c9f3ed45954217d5b4fc3a275fbe4f68f6aafcd0ed437f49299532f78421b65c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00013/source-offsets.npy': '86b5ea37cc0e6571c469cbe65a9895d6ad82c58c91dd32a7f98b4b7e72cc65e9', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00013/target-offsets.npy': '6b671e7ea6c9426f6e8054fc7cbb5433792a977296b18d55c5a4ea1e94d6598f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00014/source-offsets.npy': 'e6604b716f560f6f4589c40a19651bf9e8738db5aa5d023cb6834c7efd9f61d9', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00014/target-offsets.npy': '16034c619d288ee0da48e48724df78a7c524f17708551b603e01d715c5e75ec0', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00015/source-offsets.npy': 'c0498161727d0af351d28b1ea2d6e72d7a4a6691527a20fe98689520a145a2df', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00015/target-offsets.npy': '5e2faace300935864856d4304fcdeba5444decadb0201e0eea2ce7f5bccb8f1f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00016/source-offsets.npy': '642234cbda4e9dbbcd86df0ff7c3727fee851ac0e6150bdfcb0fcf4130ea7ac5', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00016/target-offsets.npy': 'c83fa7c56116f40a3a7ff52f569692dd808c1f6270a0785d7b835320e8df82e1', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00017/source-offsets.npy': 'b2c0c68b3a9560c7b65d8f5706f470ca796e6a3491ef860949e78960754e3f2d', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00017/target-offsets.npy': '202fcbeeb1d4be1ee7ad46eb74e0d9a1e32c04f27333872eba7431bcfaf4ed56', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00018/source-offsets.npy': '1f7423bd797f73846c5619e12f404e4920aef1f6e0386083557755d0ee236493', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00018/target-offsets.npy': 'fcb30eb1dfe40bd4cec772e0c032afed7b841d7341fc41bbb49c3ef94288e9dd', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00019/source-offsets.npy': 'bde0083f71b94ade3d0d1a18e5d41f4ba3f3634778cef0530397933f75e99856', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00019/target-offsets.npy': '9159f44c79a15502caf7651c374f9393f5724499de6fbce1a3874bf904982dd9', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00020/source-offsets.npy': '434b7bbd4bf8562aff1205408aefeacfe088570ad7e64047f92eb6839f13f8ea', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00020/target-offsets.npy': 'c48226d7f953a43efe8dd8dd8bbee1eccc825843a25ef7f920639c450bdc5712', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00021/source-offsets.npy': '3eb45cadc81f9ed0b5bcaf3a0a1981a71dbcf64b514d64a05e9ac7b6388e7135', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00021/target-offsets.npy': '47b7be3c4ab29d5eb47ab22496a48c3b103dd47181de6887f35fa66437a944eb', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00022/source-offsets.npy': 'eace981c8b274215bb084ebb29edee2994f1484d1eb80ff9b659a704f7a911c7', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00022/target-offsets.npy': 'b4ac30f251c65ed2a48ceb96de21b914892f69bfa108fd5dd1b7d4c54be189b7', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00023/source-offsets.npy': '8fb7916d25a4e6ac9005a46854a306f3d46af281e4bec05c1bfaff8967567e21', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00023/target-offsets.npy': '94852e7c24e3595c42a05f1bf96b63007c7563ed79549de0e48a9bdbf9c00936', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00024/source-offsets.npy': '026472bc53e79f19d840ce8a9bb2bbca66ceb1ab297d6f3f4fe4c9d3dcfe107a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00024/target-offsets.npy': 'c3c916b578e6b3fab5e30fe94bf65ba61ffb6ac18cc8995858bcd5e4a815c42a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00025/source-offsets.npy': 'ae8c1baa5b984fe15c2a27c29b71e04e9d3d3eff2b187f805694649b6258d4a7', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00025/target-offsets.npy': 'c29b4610fcd9431aa659874e776a0330d7ed2c61c38710c22e0e488b11f4610c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00026/source-offsets.npy': '817dfff87733067adf5af1b500088c6e1697c5e21bd5602eeaf0c10f9393e6a3', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00026/target-offsets.npy': '92e96c1a1a22025ad6885b044dff84baa9cdd4ee5363cf1e7f827de6f42e99c1', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00027/source-offsets.npy': '6f807b5626590ec61a1d04f16f15fd363484666b382dee23fefa54057ac36a44', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00027/target-offsets.npy': '821409aee804c1c8cdeac91fcb86ebe1b4811046418fad5ad4e1aeae05c2db6a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00028/source-offsets.npy': 'aed1b45f8072e70caab71f0cf9e8c447e2a8703332473c9a81dfda2ece05e127', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00028/target-offsets.npy': '45b527da607cb97620dec3f5d5b22ba9514371621f8b92032961baba7d8553c1', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00029/source-offsets.npy': '6048d60a8a75242f09058dd16d4b1d195487b101e7e36b28a316ccc49fa03377', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00029/target-offsets.npy': 'e9db3b34f4fc4ea59d4c7a21931bb7507f6b43bf7ddf83a42e052f98d531ce53', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00030/source-offsets.npy': '50715178c10f10493c8a9356ca3ca7277eba1a8a21cf10a2c0c870e7afad8888', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00030/target-offsets.npy': 'e6235d5720aaeb7ff095aa013978c9efde6acd61329c3385f2f61acffbbad925', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00031/source-offsets.npy': '6450eb41193fb04b28616f880fb1982a87dc786ab375a9dc183822693a32b4ad', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00031/target-offsets.npy': '395a5a9129a04454c87c8e240ed4c755df0661dabc6696358995bf476c6b48ef', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00032/source-offsets.npy': 'dd2e3c07f46026937c4d10ad2d8674a9fd816ae9f847e2a612c8ab896cb59069', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00032/target-offsets.npy': '94a29638599edfc67ac6dac9f19ac99f29ce0413beeeca9d6678eb587ac81f74', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00033/source-offsets.npy': '1698e88eed62089d49d85fd290b75fc76bcf97d0840897b97bb09d3b5bedecc8', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00033/target-offsets.npy': '4d6599ebce504854d0c21c825202e03bea172c6ae4b8ba3ec2270cf45337a978', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00034/source-offsets.npy': '23124f865b30d3aedc4cf783184c63a168342a329487c8295105a5334806c229', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00034/target-offsets.npy': '4aefb58e1f4a89dcd6764ea4be43bfb1a8dcd00252566af3a2e89b32d88c4419', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00035/source-offsets.npy': '5f0295803a559def3cc5d05647bb386b02857c026b658302cf530c7b235df7dd', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00035/target-offsets.npy': '6ad34889172b307b7fe2772cf6a5b9543a40d6c7838eea96ef341068c85142ce', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00036/source-offsets.npy': 'ba8441ee90f70eadea43c1fbe3df6a676f75b43a0056c72810a0c213880a7910', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00036/target-offsets.npy': '937b496d2dffa9d84c8e04330b5f32cf58751e31c9157e835f408be7be366946', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00037/source-offsets.npy': 'ee8eb5e311b63e78fef39ca038c38d18c65603ed8db60c54bc6912159dbe074c', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00037/target-offsets.npy': '2e859439e772f410406dc90fa211dc586c240e68c5fdcd23b87810b17983a0a6', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00038/source-offsets.npy': 'bb106694ccdd2e192df95c6309cc887a2b167176b98abce1d30843cd40e4424b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00038/target-offsets.npy': 'c02e7ee3c58161f9717e1f5471112c8385cce309d8b087221c5e05de28933782', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00039/source-offsets.npy': '4f678d06b84167452312546f282ac1b16dcc9df8c8d177209f6b025461040070', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00039/target-offsets.npy': '377c20859c4ce6eda5502f259af9f04d871ac3886516d8b9174e3f409d589cf3', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00040/source-offsets.npy': '6549cd37867825abefb4fff1eacfb00aaffebe85b89cdb5c99aae32c8693cb02', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00040/target-offsets.npy': '9a1271b5115acbc1fcf412ae8a2561e6e712302ccdaa4770b874b4ab8df4ec6f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00041/source-offsets.npy': 'b74b7d29cd37b823b0c8877cdf8f04a9f10a293209f8cf438a10451380745d30', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00041/target-offsets.npy': 'b0e697f9a881cb1cabef364f0cd8830c40ec17279dd61d872df540f0223e00cf', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00042/source-offsets.npy': '52c34be9c3715fad2600dae9b7e7e080f56ee375602369a4c377ca94afbb7b69', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00042/target-offsets.npy': '2bb52ffea98299c10bd851c7972485f9a51996cb11ffb60782010d769e3bbdbf', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00043/source-offsets.npy': 'f5ed489ce1d6d6ce96b6015c465a06c33b24ef9cf4fa08ab9fe99aef663ba036', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00043/target-offsets.npy': '4365682a92fed2b3f8ff55c628ac7dcd5329e5950c7428c20ea16292d8720e7f', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00044/source-offsets.npy': 'effde236d65004b699a19a11fd72634e828dc9862d59303dc90e635a048fb6d8', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00044/target-offsets.npy': '82cd34d8706847576b348b9bb7858134e4e6ef971057793cb855c4be3a6064b5', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00045/source-offsets.npy': '6484e501ee7ca0fc84164cf0da0cd64cc3d9be4eb54bc489e0986ce692663d70', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00045/target-offsets.npy': '9ff81caea774a3a43e3cc554a99e0e45b4bf05838409503d7a28529e9dad3c49', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00046/source-offsets.npy': '6835f497f886311f060a094d1db462637f5162321a8e261a4baa42d3a11a211a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00046/target-offsets.npy': '42be908011946e09c6bdde686bcd200ec7c24d9ae697e56c35e04fc7f3719c89', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00047/source-offsets.npy': 'a4157271ef39cdabbeb116d43346d6ab01ae1643b24af515fc09453b2a051fb2', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00047/target-offsets.npy': '3b41add62c4ff7096e0fb2347e96863e9d1d482fe06ff1dc5eb75f5a04ae9987', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00048/source-offsets.npy': 'e9c016526606254ff0a61fc15ce3becc9b3da3e0838fc30fdb5687619e766963', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00048/target-offsets.npy': '571eb9ed843aed60e073ff2e0fdf45a9615c1c8b715e3240f18d2c479f3a3e95', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00049/source-offsets.npy': 'db315825ec0ea6667e3810de67093b73267cd1558243afa456fcb97ba1d4bb28', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00049/target-offsets.npy': '5f88c4b9b1c0dfc938a4797af326605b255e29d67b0ae30baf8fa95a03b18e5d', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00050/source-offsets.npy': '97f220b5a7445db8da2c93904b915fe39d47c0096d46a52217ef9283171899af', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00050/target-offsets.npy': '56b76acdf2e46332dd3d2a9db64517d2fd8f158e2023e68a318f4a3e98972806', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00051/source-offsets.npy': 'e52bcc23053c467061fe7e31a3820daa47023998a21ca96da8c6a0bd42cf1480', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00051/target-offsets.npy': '01d74418e627a48af51556dd022b9cabdc194c5ac627df4f41593fd2676080be', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00052/source-offsets.npy': 'fe503d40eea599dedbba55aba52dc09723991ecb4a1292827af5846e8c9348c2', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00052/target-offsets.npy': '440928beed0129a0728d3cbbbbe24cd01a3f49ecd89845ae68f9120c77093c60', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00053/source-offsets.npy': 'cb7298ddc10ace7531b661195b268c2cebc68d89676713900ddad3359d60171b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00053/target-offsets.npy': 'bc25228b3797405064308375811db6dfdedecd475171fb7a3d9bf4d2b43eba1b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00054/source-offsets.npy': '06e8033c3779a231a7a02cddf7934d8b42dc8a3f22b6e81c860dbee71741063b', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00054/target-offsets.npy': '961568c8ec6e8285bd428fc9abf715cf5e2effddfc193da1f12648f39a516705', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00055/source-offsets.npy': 'd558786dc2ec1754810fd0f5170b847e110162b4e4d2fa74bd694045c240bbc4', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00055/target-offsets.npy': 'fe0df0f68eede72ba2da739d0a2497f992a7528f5b92ac1073ab0f00da2ccb6a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00056/source-offsets.npy': 'c47a122bba5a498bcf7149798f196d47f328380d3957f1df671bf7880b03ac42', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00056/target-offsets.npy': '83008ca48fc576fadd75863a36b60721dd2b2d2601792bb8f9f8099e52274107', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00057/source-offsets.npy': '35f44a1fcd25d9d2ca1582836707d95ca3d8a51c4e84d080f9a1779ffcc71e44', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00057/target-offsets.npy': '29134769a98548d977e5be40dc302bd466f97e7e281605a65336754927c1719a', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00058/source-offsets.npy': 'f47ad4199faabf9634cc3d971008a76abb5ef9fc2462725c9d1ad5b89f61d900', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00058/target-offsets.npy': '1aa9fa1d506500c405a998d401c6275121ec7601945d96105916a95a5eae5f3d', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00059/source-offsets.npy': '09ae7437391fc12f4bdf47b236f436e559898cc776e0b488a9cafa3899f7e1f8', 'results/kaggle-runs/nlp-hw02-compact-cache-verified/2/compact-cache/shards/part-00059/target-offsets.npy': 'c7b972fbff8f7f198d6cfb70cfc2bdb9870cb8f812e99257e6df420b97319214'}, 'test_generation_sha256': 'b1aab84c7637b6b2f877282d50c4ce0fe3b309911d9a24b10535ce5605bda7e6', 'tokenizer_sha256': '2b6fbb6426c42e9fcf82d576d53516d5b6e585c3618942f0dde6c76349856c1e'}, 'semantics': 'Random token IDs with real joint lengths. Target length includes EOS; source includes BOS/EOS. No training text embedded.'}

for name,source in SOURCES.items(): (ROOT/name).write_text(source)
(ROOT/'lengths.json').write_text(json.dumps(LENGTHS))


In [ ]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES='0',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
command=[sys.executable,'-c',"import shape_bench; shape_bench.run('lengths.json',"+repr(str(OUTPUT))+","+repr(SESSION_START)+")"]
try:
    with (OUTPUT/'run.log').open('w') as log:
        child=subprocess.Popen(command,cwd=ROOT,env=env,start_new_session=True,stdout=log,stderr=subprocess.STDOUT)
        children.add(child.pid)
        while child.poll() is None:
            if time.monotonic()-SESSION_START>=1180: kill_children(); break
            time.sleep(.2)
        child.wait(timeout=3); children.discard(child.pid)
    print('Exit',child.returncode)
    if (OUTPUT/'summary.json').exists():
        report=json.loads((OUTPUT/'summary.json').read_text()); print(report['status'],report.get('wall_s'))
    if (OUTPUT/'timings.csv').exists(): print((OUTPUT/'timings.csv').read_text())
    if child.returncode: print((OUTPUT/'run.log').read_text()[-3000:])
finally:
    kill_children(); stop.set()
